In [ ]:
from model import NetflowClassifier
from client import Client
import numpy as np
import pandas as pd
import copy
import torch
from torch.utils.data import TensorDataset

# 1. Load raw dataframe
df = pd.read_parquet("NF-UNSW-NB15-V2.parquet")

num_clients = 5 

# 2. Separate features/labels, convert to tensors
y = df["Label"].values
X = df.drop(columns =["Label", "Attack"]).values.astype(np.float32)
X = (X - X.mean(axis=0))/ (X.std(axis=0)+1e-8)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

input_dim = X.shape[1]


init_model = NetflowClassifier(input_dim)


def setup():
    rng = np.random.default_rng(seed=42)
    indices = rng.permutation(len(df))
    client_indices = np.array_split(indices, num_clients)

    clients = []
    for i in range(num_clients):
        idx = client_indices[i]
        client_dataset = TensorDataset(X[idx], y[idx])

        client = Client(client_id=i, dataset=client_dataset, input_dim=input_dim)
        clients.append(client)
    return clients


<bound method NDFrame.head of          L4_SRC_PORT  L4_DST_PORT  PROTOCOL  L7_PROTO  IN_BYTES  IN_PKTS  \
0               1305           21         6       1.0         9        1   
1               1305           21         6       1.0       261        5   
2               1305           21         6       1.0       481        9   
3               1305           21         6       1.0       701       13   
4               1305           21         6       1.0      1031       19   
...              ...          ...       ...       ...       ...      ...   
1986740        20890         5190         6       0.0      1064       12   
1986741        58663         5190         6       0.0      1064       12   
1986742        54553           80         6       7.0       994       10   
1986743        55026         8248         6       0.0      4014       68   
1986744        28987        26948         6       0.0      2246       34   

         OUT_BYTES  OUT_PKTS  TCP_FLAGS  CLIENT_TCP_FLAGS

In [3]:
from server import Server

clients = setup()
server = Server(init_model)
server.FL_algo(clients, 100)


	loss: 0.6516949534416199
	loss: 0.6205360293388367
	loss: 0.5732033848762512
	loss: 0.5078219771385193
	loss: 0.8563306331634521
************************
	loss: 0.5784258842468262
	loss: 0.5958644151687622
	loss: 0.5598567128181458
	loss: 0.5996179580688477
	loss: 0.5488768815994263
************************
	loss: 0.5656154155731201
	loss: 0.5618796944618225
	loss: 0.6023348569869995
	loss: 0.5951919555664062
	loss: 0.6075925230979919
************************
	loss: 0.5938705205917358
	loss: 0.5701524019241333
	loss: 0.5311082005500793
	loss: 0.5432936549186707
	loss: 0.5989022850990295
************************
	loss: 0.5624929070472717
	loss: 0.6284857988357544
	loss: 0.5879058241844177
	loss: 0.5824076533317566
	loss: 0.6307032108306885
************************
	loss: 0.5773798227310181
	loss: 0.5836307406425476
	loss: 0.5681590437889099
	loss: 0.5398409366607666
	loss: 0.6223329305648804
************************
	loss: 0.57505863904953
	loss: 0.5695838928222656
	loss: 0.54406416416